# 02 — Preparação dos dados

Constrói a tabela por curso, audita duplicados/nulos, trata anos impossíveis, cria split 80/20 e demonstra o Pipeline anti-leakage.

In [1]:
from pathlib import Path
import sys

# O notebook pode ser executado a partir da raiz ou da pasta notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /Users/djalma.rodrigues/projetos/99_Revisar/projeto-ia-ciencia-dados-2026


In [2]:
import json
import pandas as pd
from src.data import build_course_dataset, feature_columns
from src.preparation import prepare_and_split, TARGET_BINARY

course_data, aggregation_audit = build_course_dataset(save=True)
display(pd.Series(aggregation_audit, name="valor"))
print("Tabela curso:", course_data.shape)
print("Duplicatas de CO_CURSO:", course_data["CO_CURSO"].duplicated().sum())
display(course_data[["CO_CURSO", "N_INSCRITOS", "N_RESULTADOS_VALIDOS", "NT_GER_MEDIA_CURSO"]].head())

conflitos_atributos_curso     {'NU_ANO': 0, 'CO_IES': 0, 'CO_CATEGAD': 0, 'C...
linhas_antes_filtro_nota                                                   9812
linhas_com_nota_valida                                                     9380
duplicatas_curso_removidas                                                    0
colunas_dataset                                                              97
regra_uniao                   agregar cada arquivo por CO_CURSO antes de uni...
Name: valor, dtype: object

Tabela curso: (9380, 97)
Duplicatas de CO_CURSO: 0


,CO_CURSO,N_INSCRITOS,N_RESULTADOS_VALIDOS,NT_GER_MEDIA_CURSO
0,3,33,31.0,59.612903
1,9,42,36.0,59.377778
2,10,13,11.0,45.709091
3,12,78,78.0,69.443590
4,16,28,23.0,51.273913


### Duplicados

Repetições brutas não são removidas porque não existe identificador individual e pessoas distintas podem compartilhar os valores publicados. Depois da agregação, `CO_CURSO` é chave única; duplicatas nessa tabela são removidas/validadas.

In [3]:
train, test, metadata = prepare_and_split(save=True)
display(pd.Series({k: v for k, v in metadata.items() if not isinstance(v, list)}, name="valor"))
print("Sobreposição de cursos:", len(set(train.CO_CURSO) & set(test.CO_CURSO)))
print("Classes no treino:")
display(train[TARGET_BINARY].value_counts(normalize=True).sort_index().rename("proporção"))
print("Classes no teste:")
display(test[TARGET_BINARY].value_counts(normalize=True).sort_index().rename("proporção"))

random_state                                                                         42
test_size                                                                           0.2
min_resultados_validos_por_curso                                                     10
limiar_mediana_aprendido_no_treino                                            46.816228
target_score                                                         NT_GER_MEDIA_CURSO
target_binario                                              TARGET_ACIMA_MEDIANA_TREINO
descricao_target                      1 se a média NT_GER do curso excede a mediana ...
nao_eh                                    nota de aprovação ou conceito oficial do INEP
n_train                                                                            6178
n_test                                                                             1545
prevalencia_train                                                                   0.5
prevalencia_test                

Sobreposição de cursos: 0
Classes no treino:


TARGET_ACIMA_MEDIANA_TREINO
0    0.5
1    0.5
Name: proporção, dtype: float64

Classes no teste:


TARGET_ACIMA_MEDIANA_TREINO
0    0.491262
1    0.508738
Name: proporção, dtype: float64

### Nulos e outliers

Anos fora de 1950-2023, incluindo erros documentados pelo INEP, viram nulos; sua taxa é preservada. Idades na faixa oficial 17-89 são mantidas. Demais numéricas recebem mediana, clipping IQR e escala dentro do Pipeline ajustado no treino.

In [4]:
numeric = metadata["numeric_features"]
categorical = metadata["categorical_features"]
missing = train[numeric + categorical].isna().mean().sort_values(ascending=False)
display(missing.head(15).rename("proporção_nulos"))

q1 = train[numeric].quantile(0.25)
q3 = train[numeric].quantile(0.75)
iqr = q3 - q1
outlier_counts = ((train[numeric] < (q1 - 1.5 * iqr)) | (train[numeric] > (q3 + 1.5 * iqr))).sum()
display(outlier_counts.sort_values(ascending=False).head(15).rename("outliers_IQR_antes_pipeline"))

N_INSCRITOS                  0.0
ANOS_DESDE_FIM_EM_MEDIANA    0.0
PROP_QE_I25_B                0.0
PROP_QE_I25_A                0.0
TX_RESPOSTA_QE_I23           0.0
PROP_QE_I23_E                0.0
PROP_QE_I23_D                0.0
PROP_QE_I23_C                0.0
PROP_QE_I23_B                0.0
PROP_QE_I23_A                0.0
TX_RESPOSTA_QE_I22           0.0
PROP_QE_I22_E                0.0
PROP_QE_I22_D                0.0
PROP_QE_I22_C                0.0
PROP_QE_I22_B                0.0
Name: proporção_nulos, dtype: float64

PROP_QE_I08_G         1409
PROP_QE_I26_A         1292
PROP_QE_I26_D         1164
PROP_TURNO_1          1090
PROP_QE_I25_G          831
PROP_QE_I25_D          634
IDADE_MEDIANA          607
PROP_QE_I17_F          536
PROP_QE_I26_G          491
PROP_QE_I15_F          476
N_INSCRITOS            467
TX_RESPOSTA_QE_I17     444
TX_RESPOSTA_QE_I21     444
TX_RESPOSTA_QE_I22     444
TX_RESPOSTA_QE_I25     444
Name: outliers_IQR_antes_pipeline, dtype: int64

### Pipeline e ColumnTransformer contra vazamento

In [5]:
from src.modeling import build_preprocessor

features = numeric + categorical
preprocessor = build_preprocessor(numeric, categorical)
# fit_transform recebe exclusivamente X_train. Em modelagem, fica dentro de cada Pipeline/fold.
X_train_transformed = preprocessor.fit_transform(train[features], train[TARGET_BINARY])
X_test_transformed = preprocessor.transform(test[features])
print(preprocessor)
print("Shapes transformados:", X_train_transformed.shape, X_test_transformed.shape)
print("Features pós-encoding:", len(preprocessor.get_feature_names_out()))

ColumnTransformer(sparse_threshold=0,
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('outliers_iqr', IQRClipper()),
                                                 ('scaler', StandardScaler())]),
                                 ['N_INSCRITOS', 'ANOS_DESDE_FIM_EM_MEDIANA',
                                  'ANOS_DESDE_INICIO_GRAD_MEDIANA',
                                  'TX_ANO_FIM_EM_INVALIDO',
                                  'TX_ANO_IN_GRAD_INVALIDO', 'PROP_TURNO_1',
                                  'PROP_TURNO_2', 'PROP_TURN...
                                  'PROP_QE_I08_D', 'PROP_QE_I08_E',
                                  'PROP_QE_I08_F', 'PROP_QE_I08_G',
                                  'TX_RESPOSTA_QE_I08', 'PROP_QE_I10_A',
                                  'PROP_QE_I10_B', ...

A normalização é necessária para a Regressão Logística e inócua para as árvores; manter um pré-processador comum torna a comparação rastreável. O one-hot representa códigos nominais sem impor ordem.